# Homework 1

#### EE-556 Mathematics of Data - Fall 2026


In this homework, we consider a multiclass classification task modeled by multinomial (softmax) logistic regression. Your goal will be to analyze the estimator and its properties (convexity, existence/uniqueness), and to derive gradients/Hessians and smoothness bounds. The first part consists of theoretical questions only.


<div class="alert alert-info">
  ℹ️ <strong>Information on group based work:</strong>
</div>

- Only one group member needs to submit all five completed notebooks on behalf of the group.
- You may seek help outside your group, but each group must write its own answers.
- If the work was <span style="color: red;">unevenly distributed</span> among group members, let the TAs know.
- Only one member of the group is allowed to use AI. We will require sharing the conversation history with the AI in the form of a public link. If you use multiple conversations across the same or multiple tools please share all of them. Name the person in your group who is allowed to use AI. We encourage you to use the AI to help you understand the material, but we ask you to write the code and theory solutions by yourself. Part 5 permits AI usage as specified in its instructions.

<div style="border: 1px solid #f00; background-color: #fdd; padding: 10px; border-radius: 5px;">
  ⚠️ Do not forget: List each group member's name and SCIPER number.
</div>


Person 1 **Francesco Foglia**: || Person 1 **425511**: 


Person 2 **Antonio Angelo Polignano**: || Person 2 **436345**:


Person 3 **Name**: || Person 3 **SCIPER**:


<div style="border: 1px solid #0a0; background-color: #dfd; padding: 10px; border-radius: 5px;">
  📓 Feedback on AI use: Please use the following cell to provide feedback on the AI use in this notebook.
  
  For example, how useful were the tools to you? Which tools did you use? Did you feel like they helped you understand the material better?
</div>

*Describe your use of AI tools here.*


## 1. Multiclass Softmax Logistic Regression - 30 Points


We now model multiclass classification with classes $c \in \{1,\dots,C\}$. For each sample $(\mathbf{a}_i, b_i)$ with $\mathbf{a}_i \in \mathbb{R}^p$ and $b_i \in \{1,\dots,C\}$, let $\mathbf{X} = [\mathbf{x}_1,\dots,\mathbf{x}_C] \in \mathbb{R}^{p\times C}$ be the class weight matrix. The softmax model defines

$$
\mathbb{P}(b_i = c \mid \mathbf{a}_i) = \frac{\exp(\mathbf{a}_i^\top \mathbf{x}_c)}{\sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k)}.
$$

Assume i.i.d. samples $\{(\mathbf{a}_i,b_i)\}_{i=1}^n$. Our goal is to estimate $\mathbf{X}$ by maximum likelihood (and later with an $\ell_2$ regularizer).


__(a)__ (2 points) Show that the negative log-likelihood $f$ can be written as:

$$
\begin{aligned}
 f(\mathbf{X})
 &= - \log \mathbb{P}(b_1,\dots,b_n\mid \mathbf{a}_1,\dots,\mathbf{a}_n)\\
 &= \sum_{i=1}^n \left[ -\mathbf{a}_i^\top \mathbf{x}_{b_i} + \log \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k) \right].
\end{aligned}
$$



By i.i.d, the joint law can be computed as
$$
\begin{align*}
f(\mathbf{X}) 
&= - \log \prod_{i=1}^n \mathbb{P}(b_i \mid \mathbf{a_i}) \\
&= - \sum _{i=1}^n \log \mathbb{P}(b_i \mid \mathbf{a_i}) 
\end{align*}
$$
We now look at each individual term

$$
\begin{align*}
\log \mathbb{P}(b_i \mid \mathbf{a}_i) 
&= \log \left( \frac{\exp(\mathbf{a}_i^\top \mathbf{x}_{b_i})}{\sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k)}\right) \\
&= \log(\exp(\mathbf{a}_i^\top \mathbf{x}_{b_i})) - \log \left(\sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k)\right) \\
&= \mathbf{a}_i^\top \mathbf{x}_{b_i} - \log \left(\sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k)\right)
\end{align*}
$$

by substituting in the first equality obtain the requested identity.

__(b)__ (3 points) Show that $\mathbf{u} \mapsto \log\!\left(\sum_{k=1}^C e^{u_k}\right)$ is convex on $\mathbb{R}^C$. Then, show that $f(\mathbf{X})$ is convex.


Hint: use Jensen's inequality.

We prove convexity by considering the standard algebraic definition. Let $g$ be the map in definition and $\lambda \in (0, 1)$
$$
\begin{align*}
g(\lambda \mathbf{x} + (1 - \lambda) \mathbf{y})
&= \log(\sum_{k=1}^C e^{\lambda x_k + (1 - \lambda) y_k}) \\
&= \log(\sum_{k=1}^C e^{\lambda x_k} e^{(1 - \lambda) y_k}) \\
&\leq \log \left(\sum_{k=1}^C e^{x_k} \right)^\lambda \left(\sum_{k=1}^C e^{y_k} \right)^{1 - \lambda} \\
&= \log \left(\sum_{k=1}^C e^{x_k} \right)^\lambda + \log \left(\sum_{k=1}^C e^{y_k} \right)^{1 - \lambda} \\
&= \lambda \log\left(\sum_{k=1}^C e^{x_k} \right) + (1 - \lambda) \log \left(\sum_{k=1}^C e^{y_k} \right) \\
&= \lambda g(\mathbf{x}) + (1 - \lambda) g(\mathbf{y}) \\ \\
\implies g(\lambda \mathbf{x} + (1 - \lambda) \mathbf{y}) &\leq \lambda g(\mathbf{x}) + (1 - \lambda) g(\mathbf{y})
\end{align*}
$$
where the inequality step is using Holder's inequality with exponents $p = \frac{1}{\lambda}, q = \frac{1}{1 - \lambda}$ 
and the monotonicity of the logarithm.

For each sample $i$ define the linear map $L_i:\mathbb{R}^{p\times C}\to\mathbb{R}^C$, $L_i(\mathbf{X}) 
= \mathbf{X}^\top\mathbf{a}_i = 
(\mathbf{a}_i^\top\mathbf{x}_1,\dots,\mathbf{a}_i^\top\mathbf{x}_C)^\top$. Then
$$
f(\mathbf{X}) = \sum_{i=1}^n \Big[ -\mathbf{a}_i^\top \mathbf{X}\mathbf{e}_{b_i} + 
g\big(L_i(\mathbf{X})\big) \Big].
$$
The first term is linear in $\mathbf{X}$, hence convex. The second is convex because it is a convex 
function composed with a linear map: for $\lambda\in[0,1]$, $g(L_i(\lambda\mathbf{X}+(1-
\lambda)\mathbf{Y})) = g(\lambda L_i(\mathbf{X})+(1-\lambda)L_i(\mathbf{Y})) \le \lambda 
g(L_i(\mathbf{X})) + (1-\lambda) g(L_i(\mathbf{Y}))$. A sum of convex functions is convex, so $f$ is 
convex.

Since the negative log-likelihood $f$ is convex, every local minimizer is a global minimizer. The maximum likelihood estimation problem is
$$
\mathbf{X}^\star_{ML} = \arg\min_{\mathbf{X} \in \mathbb{R}^{p\times C}} f(\mathbf{X})
$$

But does $f$ always attain its infimum? The following three questions examine this issue.


__(c)__ (1 point) Explain the difference between infima and minima. Give an example of a convex function on $\mathbb{R}$ that does not attain its infimum.


The infimum of a set is as the largest possible lower bound of its elements and is always well defined. It takes values in the extended real numbers and it is equal to the minimum whenever the latter is well defined. In terms of functions, the infimum is the largest value that is smaller than or equal to any value in the image.

The infimum might not be part of the image, or it could be $-\infty$. Whenever the infimum is finite and it is attained (i.e. it is part of the image), then the minimum is well defined and is the infimum itself.

The function $f(x) = e^x$ has the requested properties. Its infimum is $0$, it is convex, but there is no finite real value such that $f(x) = 0$. 

__(d)__ (2 points) Assume there exists $\mathbf{X}_0 \in \mathbb{R}^{p\times C}$ such that for all $i$,
$$
\mathbf{a}_i^\top \mathbf{x}_{0, b_i} - \max_{k \neq b_i} \mathbf{a}_i^\top \mathbf{x}_{0,k} > 0.
$$
This is called one-versus-all complete separation in multiclass settings. Give a geometric interpretation (e.g., for $p=2$) and explain why the name is appropriate.


It's helpful to look at the equations by grouping samples of the same class and factoring. 
$$
\begin{cases}
\mathbf{a}_{c,1}^\top (\mathbf{x}_{0, c} - \mathbf{x}_{0,k}) > 0 \quad \text{for all } k \neq c\\
\mathbf{a}_{c,2}^\top (\mathbf{x}_{0, c} - \mathbf{x}_{0,k}) > 0 \quad \text{for all } k \neq c\\
... \\
\end{cases}
$$ 
In geometrical terms, these equations are describing $C - 1$ hyperplanes (passing through the origin) that have all points of class $c$ on the same side (respectively the hyperplane passing through the origin that is orthogonal to $(\mathbf{x}_{0, c} - \mathbf{x}_{0,k})$). For samples of a fixed class $k \neq c$, we can look at the same equation and obtain
$$
\begin{cases}
\mathbf{a}_{k,1}^\top (\mathbf{x}_{0, k} - \mathbf{x}_{0,c}) > 0 \\
\mathbf{a}_{k,2}^\top (\mathbf{x}_{0, k} - \mathbf{x}_{0,c}) > 0 \\
... \\
\end{cases}
$$  
So the hyperplane defined by $(\mathbf{x}_{0, c} - \mathbf{x}_{0,k})$ is strictly separating classes $c$ and $k$. We now have pairwise separation, to get a "one-versus-all" separation we consider the $C-1$ $p$-dimensional regions of space that separate class $c$ from each of the other classes. If we intersect all such regions, we obtain a region of space (still $p$-dimensional) that separates class $c$ from all other classes!

Since the derivation is independent of the chosen class $c$ and the separations are defined by the previous hyperplanes, we can see that the inequalities are really defining a partitioning of space in $C$ $p$-dimensional regions, so that samples of the same class are in the same region. 

In the specific case of $p = 2$ we can visualize such regions and see that (as they are defined by intersecting half-planes with border always passing through the origins) the plane is partitioned in $C$ different half-cones. In this case we can visualize what "one-versus-all" means, as each region is strictly separating a class from all the other ones.

Complete separation allows perfect classification of the training data. However, as you will show next, it prevents the existence of a finite maximum likelihood estimate.


__(e)__ (3 points) In a one-versus-all complete separation setting (as in (d)), prove that $f$ does not attain its minimum. Hint: consider $f(\alpha \mathbf{X}_0)$ as $\alpha \to +\infty$ and compare it to $f(\mathbf{X}_0)$.


We first prove that the function $f$ is strictly positive for any $\mathbf{X}$.
$$
\begin{align*}
f(\mathbf{X}) 
&= \sum_{i=1}^n \left[ -\mathbf{a}_i^\top \mathbf{x}_{b_i} + \log \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k) \right] \\
&> \sum_{i=1}^n \left[ -\mathbf{a}_i^\top \mathbf{x}_{b_i} + \log \exp(\mathbf{a}_i^\top \mathbf{x}_{b_i}) \right] \\
&= \sum_{i=1}^n \left[ -\mathbf{a}_i^\top \mathbf{x}_{b_i} + \mathbf{a}_i^\top \mathbf{x}_{b_i} \right] \\
\implies f(\mathbf{X}) &> 0 
\end{align*}
$$
where the inequality step is justified by positivity of the exponential function.

We now aim to prove that the infimum is $0$, by finding a sequence of values that has limit in $0$, because we know that it cannot be attained.
$$
\begin{align*}
f(\alpha X_0) 
&= \sum_{i=1}^n \left[ -\alpha \mathbf{a}_i^\top \mathbf{x}_{0, b_i} + \log \sum_{k=1}^C \exp(\alpha \mathbf{a}_i^\top \mathbf{x}_{0, k}) \right] \\
&= \sum_{i=1}^n \left[ \log \exp (-\alpha \mathbf{a}_i^\top \mathbf{x}_{0, b_i}) + \log \sum_{k=1}^C \exp(\alpha \mathbf{a}_i^\top \mathbf{x}_{0, k}) \right] \\
&= \sum_{i=1}^n \left[ \log \left( \exp (-\alpha \mathbf{a}_i^\top \mathbf{x}_{0, b_i}) \left(\sum_{k=1}^C \exp(\alpha \mathbf{a}_i^\top \mathbf{x}_{0, k}) \right)\right) \right] \\
&= \sum_{i=1}^n \left[ \log \left(\sum_{k=1}^C \exp \left( -\alpha \mathbf{a}_i^\top \mathbf{x}_{0, b_i} + \alpha \mathbf{a}_i^\top \mathbf{x}_{0, k} \right) \right) \right] \\
&= \sum_{i=1}^n \left[ \log \left( \sum_{k=1}^C \exp \left(\alpha  \mathbf{a}_i^\top (-\mathbf{x}_{0, b_i} + \mathbf{x}_{0, k}) \right) \right) \right] \\
&= \sum_{i=1}^n \left[ \log (1 + \sum_{k \neq b_i} \exp (\alpha d_i^k)) \right] \quad \text{where } d_i^k :=  \mathbf{a}_i^\top (-\mathbf{x}_{0, b_i} + \mathbf{x}_{0, k}) < 0\\
\end{align*}
$$
where the strict negativity of $d_i^k$ is implied by the one-above-all separation bound.


By continuity of the logarithm and the exponential and $\lim_{\alpha \to \infty} \exp(\alpha d_i^k) = 0$ (as we have $d_i^k < 0$) we can conclude
$$
\lim_{\alpha \to \infty} f(\alpha X_0) = 0 
$$
using the fact that the sum is over a finite amount of terms.

Therefore the infimum of $f$ is 0 but it cannot be finitely attained as a minimum.

We resolve this issue by adding a regularizer. Consider the regularized function

$$
 f_\mu(\mathbf{X}) = f(\mathbf{X}) + \frac{\mu}{2} \|\mathbf{X}\|_F^2, \quad \mu > 0.
$$

__(f)__ (2 points) Show that the gradient with respect to $\mathbf{X}$ of $f_\mu$ can be expressed as
$$
 \nabla_{\mathbf{X}} f_\mu(\mathbf{X}) = \sum_{i=1}^n \mathbf{a}_i \big( \mathbf{p}_i - \mathbf{e}_{b_i} \big)^\top + \mu \mathbf{X},
$$
where $\mathbf{e}_{b_i} \in \mathbb{R}^C$ is the [one-hot vector](https://en.wikipedia.org/wiki/One-hot) for class $b_i$, $\mathbf{p}_i \in \mathbb{R}^C$ has entries $p_{i,c} = \mathbb{P}(b_i=c\mid \mathbf{a}_i)$ under the softmax model, and $\mathbf{a}_i(\mathbf{p}_i - \mathbf{e}_{b_i})^\top\in\mathbb{R}^{p\times C}$ denotes the outer product.



We consider the regularized objective $f_\mu(\mathbf{X}) = f(\mathbf{X}) + \frac{\mu}{2} \|\mathbf{X}\|_F^2$ and study the gradient of the three terms separately:
$$
\sum_{i=1}^n \mathbf{a}_i \mathbf{p}_i^\top, \quad -\sum_{i=1}^n \mathbf{a}_i \mathbf{e}_{b_i}^\top, \quad \mu \mathbf{X}
$$

1. **Regularization term:**  
   The regularizer $\frac{\mu}{2} \|\mathbf{X}\|_F^2$ behaves as the standard Euclidean 2-norm squared over all matrix entries, so its gradient with respect to $\mathbf{X}$ is directly $\mu \mathbf{X}$.

2. **Linear term:**  
   For the term $\sum_{i=1}^n -\mathbf{a}_i^\top \mathbf{x}_{b_i}$, we differentiate with respect to each column $\mathbf{x}_c$ for $c \in \{1, \dots, C\}$:
   $$
   \nabla_{\mathbf{x}_c} \left( -\mathbf{a}_i^\top \mathbf{x}_{b_i} \right) = \begin{cases} -\mathbf{a}_i & \text{if } c = b_i, \\ \mathbf{0} & \text{if } c \neq b_i \end{cases} = -\mathbf{a}_i (\mathbf{e}_{b_i})_c
   $$
   Arranging these column vectors for $c = 1, \dots, C$ yields the outer product $-\mathbf{a}_i \mathbf{e}_{b_i}^\top$. Summing over all $n$ samples gives:
   $$
   -\sum_{i=1}^n \mathbf{a}_i \mathbf{e}_{b_i}^\top
   $$

3. **Log-sum-exp term:**  
   For the logarithmic term in $f$, we compute the gradient with respect to $\mathbf{X}$ column-by-column. For a fixed sample $i$ and column $c \in \{1, \dots, C\}$:
   $$
   \begin{align*}
   \nabla_{\mathbf{x}_c} \left( \log \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k) \right)
   &= \frac{1}{\sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k)} \nabla_{\mathbf{x}_c} \left( \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k) \right) \\
   &= \frac{\exp(\mathbf{a}_i^\top \mathbf{x}_c)}{\sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k)} \mathbf{a}_i \\
   &= p_{i,c} \mathbf{a}_i
   \end{align*}
   $$

   Arranging these columns for $c = 1, \dots, C$ into a matrix gives the outer product for sample $i$:
   $$
   \left[ p_{i,1} \mathbf{a}_i, \; p_{i,2} \mathbf{a}_i, \; \dots, \; p_{i,C} \mathbf{a}_i \right] = \mathbf{a}_i \mathbf{p}_i^\top
   $$

   Summing over all $n$ samples yields:
   $$
   \sum_{i=1}^n \mathbf{a}_i \mathbf{p}_i^\top
   $$

Combining the three terms gives the full gradient:
$$
\nabla_{\mathbf{X}} f_\mu(\mathbf{X}) = \sum_{i=1}^n \mathbf{a}_i (\mathbf{p}_i - \mathbf{e}_{b_i})^\top + \mu \mathbf{X}
$$
In entry-wise coordinates, the entry at row $j \in \{1, \dots, p\}$ and column $c \in \{1, \dots, C\}$ is:
$$
[\nabla_{\mathbf{X}} f_\mu(\mathbf{X})]_{j,c} = \sum_{i=1}^n a_{i,j} (p_{i,c} - (\mathbf{e}_{b_i})_c) + \mu \mathbf{X}_{j,c}
$$

__(g)__ (4 points) Use row-wise vectorization of $\mathbf{X}$, obtained by stacking its rows one after another into a vector in $\mathbb{R}^{pC}$. Show that the Hessian of $f_\mu$ in these coordinates can be written as
<a id="eq-softmax-hessian"></a>

$$
 \nabla^2 f_\mu(\mathbf{X}) = \sum_{i=1}^n (\mathbf{a}_i\mathbf{a}_i^\top) \otimes \big( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top \big) + \mu \mathbf{I}_{pC},\tag{1}
$$
where $\otimes$ is the Kronecker product, and $\operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top$ is the softmax Jacobian, which is positive semidefinite.


To compute the Hessian $\nabla^2 f_\mu(\mathbf{X})$, we use the row-wise vectorization $\mathbf{x} = \operatorname{vec}_{\text{row}}(\mathbf{X}) \in \mathbb{R}^{pC}$ and differentiate the gradient obtained in (f):
$$
\nabla_{\mathbf{X}} f_\mu(\mathbf{X}) = \sum_{i=1}^n \mathbf{a}_i \mathbf{p}_i^\top - \sum_{i=1}^n \mathbf{a}_i \mathbf{e}_{b_i}^\top + \mu \mathbf{X}
$$

We differentiate the three terms with respect to $\mathbf{x}$ separately.

1. **One-Hot Term**

The term $-\sum_{i=1}^n \mathbf{a}_i \mathbf{e}_{b_i}^\top$ is constant with respect to $\mathbf{X}$, so its derivative is zero:
$$
\frac{\partial}{\partial \mathbf{x}} \operatorname{vec}_{\text{row}}\left(-\sum_{i=1}^n \mathbf{a}_i \mathbf{e}_{b_i}^\top\right) = \mathbf{0}_{pC \times pC}
$$

2. **Regularization Term**

The regularization term $\mu \mathbf{X}$ is linear in $\mathbf{X}$. In vectorized coordinates $\mathbf{x} \in \mathbb{R}^{pC}$, its row-wise vectorization is simply $\mu \mathbf{x}$. Differentiating directly with respect to $\mathbf{x}$ yields:
$$
\frac{\partial}{\partial \mathbf{x}} (\mu \mathbf{x}) = \mu \mathbf{I}_{pC}
$$

3. **Outer Product Term via Projections and Direct Quotient Rule**

For a given sample $i$, consider the matrix term $\mathbf{a}_i \mathbf{p}_i^\top \in \mathbb{R}^{p \times C}$.

Let $\mathbf{u}_k \in \mathbb{R}^p$ denote the $k$-th canonical basis vector of $\mathbb{R}^p$ for $k \in \{1, \dots, p\}$. The $k$-th row of $\mathbf{a}_i \mathbf{p}_i^\top$ is extracted by:
$$
\mathbf{u}_k^\top (\mathbf{a}_i \mathbf{p}_i^\top) = (\mathbf{u}_k^\top \mathbf{a}_i) \mathbf{p}_i^\top = a_{i,k} \mathbf{p}_i^\top \in \mathbb{R}^{1 \times C}
$$

Let $\mathbf{P}_k: \mathbb{R}^{pC} \to \mathbb{R}^C$ be the projection matrix that extracts the $k$-th block of size $C$ from the row-stacked vector. Its transpose $\mathbf{P}_k^\top: \mathbb{R}^C \to \mathbb{R}^{pC}$ is an immersion matrix that places a vector from $\mathbb{R}^C$ into the $k$-th block of $\mathbb{R}^{pC}$ (leaving zeros elsewhere). Using the canonical vectors, $\mathbf{P}_k^\top = \mathbf{u}_k \otimes \mathbf{I}_C$.

The row-wise vectorization of $\mathbf{a}_i \mathbf{p}_i^\top$ is the sum of these immersed rows:
$$
\operatorname{vec}_{\text{row}}(\mathbf{a}_i \mathbf{p}_i^\top) = \sum_{k=1}^p \mathbf{P}_k^\top (a_{i,k} \mathbf{p}_i) = \sum_{k=1}^p a_{i,k} \mathbf{P}_k^\top \mathbf{p}_i
$$

Let the entries of $\mathbf{X}$ be indexed by row $m \in \{1, \dots, p\}$ and column $c' \in \{1, \dots, C\}$. The $c$-th component of $\mathbf{p}_i$ is:
$$
p_{i,c} = \frac{N_c}{D} = \frac{\exp(\mathbf{a}_i^\top \mathbf{x}_c)}{\sum_{l=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_l)} = \frac{\exp\left(\sum_{r=1}^p a_{i,r} \mathbf{X}_{r,c}\right)}{\sum_{l=1}^C \exp\left(\sum_{r=1}^p a_{i,r} \mathbf{X}_{r,l}\right)}
$$

We differentiate $p_{i,c}$ with respect to $\mathbf{X}_{m, c'}$ using the quotient rule:
$$
\frac{\partial p_{i,c}}{\partial \mathbf{X}_{m, c'}} = \frac{\frac{\partial N_c}{\partial \mathbf{X}_{m, c'}} D - N_c \frac{\partial D}{\partial \mathbf{X}_{m, c'}}}{D^2}
$$

- The denominator $D$ contains all coordinates of $\mathbf{X}$. Its derivative is:
  $$
  \frac{\partial D}{\partial \mathbf{X}_{m, c'}} = a_{i,m} \exp(\mathbf{a}_i^\top \mathbf{x}_{c'})
  $$
- The numerator $N_c$ depends on $\mathbf{X}_{m, c'}$ only when $c = c'$:
  $$
  \frac{\partial N_c}{\partial \mathbf{X}_{m, c'}} = \delta_{c, c'} a_{i,m} \exp(\mathbf{a}_i^\top \mathbf{x}_c)
  $$

Substituting both derivatives into the quotient rule: 

**When $c = c'$ (numerator is involved):** The first term contains a factor of $D$, which cancels one power of $D$ in the denominator:
   $$
   \frac{\delta_{c, c'} a_{i,m} \exp(\mathbf{a}_i^\top \mathbf{x}_c) D}{D^2} = a_{i,m} \delta_{c, c'} \frac{\exp(\mathbf{a}_i^\top \mathbf{x}_c)}{D} = a_{i,m} \delta_{c, c'} p_{i,c} = a_{i,m} [\operatorname{Diag}(\mathbf{p}_i)]_{c, c'}
   $$
**The second term (denominator derivative):** Produces a negative product of two exponentials divided by $D^2$:
   $$
   -\frac{\exp(\mathbf{a}_i^\top \mathbf{x}_c) \left[a_{i,m} \exp(\mathbf{a}_i^\top \mathbf{x}_{c'})\right]}{D^2} = - a_{i,m} \frac{\exp(\mathbf{a}_i^\top \mathbf{x}_c)}{D} \frac{\exp(\mathbf{a}_i^\top \mathbf{x}_{c'})}{D} = - a_{i,m} p_{i,c} p_{i, c'} = - a_{i,m} [\mathbf{p}_i \mathbf{p}_i^\top]_{c, c'}
   $$

Combining both terms gives:
$$
\frac{\partial p_{i,c}}{\partial \mathbf{X}_{m, c'}} = a_{i,m} \left( \delta_{c, c'} p_{i,c} - p_{i,c} p_{i, c'} \right) = a_{i,m} [\operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i \mathbf{p}_i^\top]_{c, c'}
$$

In block form, the derivative of the vector $\mathbf{p}_i \in \mathbb{R}^C$ with respect to the $m$-th row block of $\mathbf{x}$ is:
$$
\frac{\partial \mathbf{p}_i}{\partial \mathbf{x}_m} = a_{i,m} \left( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i \mathbf{p}_i^\top \right)
$$
Using the projection matrices $\mathbf{P}_m$, the derivative with respect to the entire vectorized vector $\mathbf{x} \in \mathbb{R}^{pC}$ is:
$$
\frac{\partial \mathbf{p}_i}{\partial \mathbf{x}} = \sum_{m=1}^p \frac{\partial \mathbf{p}_i}{\partial \mathbf{x}_m} \mathbf{P}_m = \sum_{m=1}^p a_{i,m} \left( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i \mathbf{p}_i^\top \right) \mathbf{P}_m
$$

Now we differentiate the full immersed vector $\sum_{k=1}^p a_{i,k} \mathbf{P}_k^\top \mathbf{p}_i$ with respect to $\mathbf{x}$:
$$
\begin{align*}
\frac{\partial}{\partial \mathbf{x}} \operatorname{vec}_{\text{row}}(\mathbf{a}_i \mathbf{p}_i^\top)
&= \sum_{k=1}^p a_{i,k} \mathbf{P}_k^\top \frac{\partial \mathbf{p}_i}{\partial \mathbf{x}} \\
&= \sum_{k=1}^p \sum_{m=1}^p a_{i,k} a_{i,m} \mathbf{P}_k^\top \left( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i \mathbf{p}_i^\top \right) \mathbf{P}_m
\end{align*}
$$

Recalling that $\mathbf{P}_k^\top = \mathbf{u}_k \otimes \mathbf{I}_C$ and $\mathbf{P}_m = \mathbf{u}_m^\top \otimes \mathbf{I}_C$, the mixed-product property of the Kronecker product yields:
$$
\mathbf{P}_k^\top \left( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i \mathbf{p}_i^\top \right) \mathbf{P}_m = (\mathbf{u}_k \mathbf{u}_m^\top) \otimes \left( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i \mathbf{p}_i^\top \right)
$$

Substituting this into the double summation:
$$
\begin{align*}
\sum_{k=1}^p \sum_{m=1}^p a_{i,k} a_{i,m} (\mathbf{u}_k \mathbf{u}_m^\top) \otimes \left( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i \mathbf{p}_i^\top \right)
&= \left( \sum_{k=1}^p \sum_{m=1}^p a_{i,k} a_{i,m} \mathbf{u}_k \mathbf{u}_m^\top \right) \otimes \left( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i \mathbf{p}_i^\top \right) \\
&= (\mathbf{a}_i \mathbf{a}_i^\top) \otimes \left( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i \mathbf{p}_i^\top \right)
\end{align*}
$$

Summing over all $n$ samples and adding the regularization contribution gives the complete Hessian:
$$
\nabla^2 f_\mu(\mathbf{X}) = \sum_{i=1}^n (\mathbf{a}_i \mathbf{a}_i^\top) \otimes \left( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i \mathbf{p}_i^\top \right) + \mu \mathbf{I}_{pC}
$$

__(h)__ (2 points) Show that $f_\mu$ is $\mu$-strongly convex.


Since $f_\mu$ is twice differentiable, $\mu$-strong convexity is equivalent to $\nabla^2 
f_\mu(\mathbf{X}) - \mu \mathbf{I}_{pC} \succeq \mathbf{0}$ for all $\mathbf{X}$. This means
$$
\sum_{i=1}^n (\mathbf{a}_i \mathbf{a}_i^\top) \otimes \left( \operatorname{Diag}(\mathbf{p}_i) - 
\mathbf{p}_i \mathbf{p}_i^\top \right) \succeq \mathbf{0}.
$$
Both factors of each Kronecker product are symmetric positive semidefinite. For 
$\mathbf{a}_i\mathbf{a}_i^\top$: $\mathbf{v}^\top\mathbf{a}_i\mathbf{a}_i^\top\mathbf{v} = 
(\mathbf{a}_i^\top\mathbf{v})^2 \ge 0$. For the softmax Jacobian: $\mathbf{v}^\top(\operatorname{Diag}
(\mathbf{p}_i)-\mathbf{p}_i\mathbf{p}_i^\top)\mathbf{v} = \sum_c p_{i,c}v_c^2 - \big(\sum_c 
p_{i,c}v_c\big)^2 \ge 0$ by Jensen, since $\mathbf{p}_i$ is a probability vector. If 
$\mathbf{A},\mathbf{B}$ are symmetric, $\mathbf{A}\otimes\mathbf{B}$ is symmetric and its eigenvalues 
are exactly the products $\lambda_j(\mathbf{A})\lambda_l(\mathbf{B})$. These are nonnegative when 
$\mathbf{A},\mathbf{B}\succeq\mathbf{0}$, so each term is positive semidefinite, and so is their sum. 
Hence $\nabla^2 f_\mu(\mathbf{X}) \succeq \mu\mathbf{I}_{pC}$ and $f_\mu$ is $\mu$-strongly convex.

__(i)__ (2 points) Is it possible for a strongly convex function to not attain its minimum? Justify your reasoning (you may assume the domain is $\mathbb{R}^{p\times C}$).


We will show that, assuming the domain is $\mathbb{R}^N$, every strongly convex function attains its infimum (and therefore its minimum). 

Let $\mathbf{x_0} \in \mathbb{R}^N$ and let $\mathbf{v} \in \partial f(\mathbf{x_0})$ be a subgradient 
$$
\begin{align*}
f(\mathbf{x}) 
&\geq f(\mathbf{x_0}) + < v, \mathbf{x} - \mathbf{x_0} > + \frac{\mu}{2}\|\mathbf{x} - \mathbf{x_0}\|_2^2 \\
&\geq f(\mathbf{x_0}) + \|\mathbf{x} - \mathbf{x_0}\|_2 \left( \frac{\mu}{2}\|\mathbf{x} - \mathbf{x_0}\|_2 - \|v\|_2 \right) \quad \text{by Cauchy-Schwartz}
\end{align*}
$$
where the first inequality is justified by convexity of $h(\mathbf{x}) = f(\mathbf{x}) - \frac{\mu}{2}\|\mathbf{x}\|_2^2$.

Whenever $\|\mathbf{x} - \mathbf{x_0}\|_2 > \frac{2}{\mu}\|v\|_2$, the term in parentheses is positive, giving $f(\mathbf{x}) > f(\mathbf{x_0})$. Consider the closed, bounded ball:

$$\mathcal{B} = \left\{ \mathbf{x} \in \mathbb{R}^N : \|\mathbf{x} - \mathbf{x_0}\|_2 \le \frac{2}{\mu}\|v\|_2 + 1 \right\}$$

by the previous observation the infimum over all of $\mathbb{R}^N$ must lie inside $\mathcal{B}$:

$$\inf_{\mathbf{x} \in \mathbb{R}^N} f(\mathbf{x}) = \inf_{\mathbf{x} \in \mathcal{B}} f(\mathbf{x})$$

Because $f$ is convex on $\mathbb{R}^N$, it is continuous. By the Weierstrass theorem, a continuous function on a closed and bounded set attains its minimum, meaning there is some $x^* \in \mathcal{B}$ with $f(x^*) = \inf_{x \in \mathcal{B}} f(\mathbf{x})$.

Lastly, strong convexity implies strict convexity, which guarantees at most one global minimizer. Hence, $x^*$ is the unique global minimum, and the infimum is attained.

We will now show that $f_\mu$ is smooth, i.e., $\nabla f_\mu$ is L-Lipschitz with respect to the Frobenius norm, with a simple conservative bound
$$
 L = \|\mathbf{A}\|_F^2 + \mu.
$$
where
$$
 \mathbf{A} = \begin{bmatrix}
  \leftarrow &  \mathbf{a}_1^\top & \rightarrow \\
  \leftarrow &  \mathbf{a}_2^\top & \rightarrow \\
   &  \ldots &  \\
  \leftarrow &  \mathbf{a}_n^\top & \rightarrow \\
 \end{bmatrix}.
$$
(You may use that the operator norm of the softmax Jacobian is bounded by $1/2$, and a looser bound $\le 1$ is acceptable for grading.)

Hint: check the properties of the spectral norm with respect to dot product, Kronecker product, and outer product.

(4 points for all three questions)


__(j-1)__ Show that $\lambda_{\max}(\mathbf{a}_i\mathbf{a}_i^T) = \left\| \mathbf{a}_i\right\|_2^2$, where $\lambda_{\max}(\cdot)$ denotes the largest eigenvalue.


As $\mathbf{a}_i \mathbf{a}_i^\top$ is a tensor product of vectors, it only allows for $1$ non-zero eigenvalue, so any non-zero eigenvalue we can find must coincide with the spectral radius. We can prove the above statement by testing against $\mathbf{a}_i$ and using associativity.

$$
\begin{align*}
(\mathbf{a}_i \mathbf{a}_i^\top) \mathbf{a}_i
&= \mathbf{a}_i (\mathbf{a}_i^\top \mathbf{a}_i) \\
&= \mathbf{a}_i \|\mathbf{a}_i\|_2^2  
\end{align*}
$$
so $\|\mathbf{a}_i\|_2^2$ is an eigenvalue and the proof is complete by the previous observations.

__(j-2)__ Using [(1)](#eq-softmax-hessian), show that $\lambda_{\max}(\nabla^2 f_\mu(\mathbf{X})) \leq \sum_{i=1}^{n} \|\mathbf{a}_i\|_2^2 + \mu$.

By the properties of the Kronecker product, we know that all eigenvalues of the terms
$$
(\mathbf{a}_i \mathbf{a}_i^\top) \otimes \left( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i \mathbf{p}_i^\top \right) \quad \text{for } i = 1, \dots, n
$$
are the products of the eigenvalues of the original matrices. Knowing that the largest eigenvalue is equal to the 2-norm on positive semi-definite matrices, we can use the triangle inequality, the previous observation  and the result in (j-1) to get the following bound
$$
\begin{align*}
\lambda_{\max}(\sum_{i=1}^n (\mathbf{a}_i \mathbf{a}_i^\top) \otimes \left( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i \mathbf{p}_i^\top \right))
&= \|\sum_{i=1}^n (\mathbf{a}_i \mathbf{a}_i^\top) \otimes \left( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i \mathbf{p}_i^\top \right)\|_2 \\
&\leq \sum_{i=1}^n \|(\mathbf{a}_i \mathbf{a}_i^\top) \otimes \left( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i \mathbf{p}_i^\top \right)\|_2 \\
&\leq \sum_{i=1}^n \|(\mathbf{a}_i \mathbf{a}_i^\top)\|_2 \cdot \|\left( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i \mathbf{p}_i^\top \right)\|_2 \\
&= \sum_{i=1}^n \|\mathbf{a}_i\|_2^2 \cdot \|\left( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i \mathbf{p}_i^\top \right)\|_2 \\
\end{align*}
$$
where the positive semi-definiteness of all the appearing terms has been established in the solution of (h).

From (g) we know that $\operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i \mathbf{p}_i^\top$ is positive semi-definite and, since $\mathbf{p}_i$ is a probability vector (with non-negative entries), the rank-1 outer product $\mathbf{p}_i \mathbf{p}_i^\top$ is also positive semi-definite from the observations in (j-1). Hence, the following holds
$$
\mathbf{0} \preceq \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i \mathbf{p}_i^\top \preceq \operatorname{Diag}(\mathbf{p}_i)
\implies \|\operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i \mathbf{p}_i^\top\|_2 \le \|\operatorname{Diag}(\mathbf{p}_i)\|_2
$$

Since $\mathbf{p}_i$ is a probability vector, its entries satisfy $0 \le p_{ik} \le 1$, therefore
$$
\|\operatorname{Diag}(\mathbf{p}_i)\|_2 = \max_k p_{ik} \le 1
$$

Substituting this back into the earlier chain of inequalities yields:
$$
\lambda_{\max}\left(\sum_{i=1}^n (\mathbf{a}_i \mathbf{a}_i^\top) \otimes \left( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i \mathbf{p}_i^\top \right)\right) \le \sum_{i=1}^n \|\mathbf{a}_i\|_2^2
$$

To conclude, consider the whole Hessian and bound again using the triangle inequality
$$
\begin{align*}
\lambda_{\max}(\nabla^2 f_\mu(\mathbf{X})) 
&= \left\|\sum_{i=1}^n (\mathbf{a}_i \mathbf{a}_i^\top) \otimes \left( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i \mathbf{p}_i^\top \right) + \mu I \right\|_2 \\
&\leq \left\| \sum_{i=1}^n (\mathbf{a}_i \mathbf{a}_i^\top) \otimes \left( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i \mathbf{p}_i^\top \right) \right\|_2 + \|\mu I\|_2 \\
&=\lambda_{\max}\left( \sum_{i=1}^n (\mathbf{a}_i \mathbf{a}_i^\top) \otimes \left( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i \mathbf{p}_i^\top \right) \right) + \mu \\
&\leq \sum_{i=1}^n \|\mathbf{a}_i\|_2^2 + \mu
\end{align*}
$$
which is the desired bound.

__(j-3)__ Conclude that $f_\mu$ is $L$-smooth for $L = \|\mathbf{A}\|_F^2 + \mu$.


We will show that $f_\mu$ is $L$-smooth with $L = \|\mathbf{A}\|_F^2 + \mu$ by using the fact that, for a twice-differentiable function, $L$-smoothness is equivalent to the following Loewner ordering
$$
\nabla^2 f_\mu(\mathbf{X}) \preceq L \mathbf{I} \quad \forall \mathbf{X}
$$

Since the Hessian $\nabla^2 f_\mu(\mathbf{X})$ is symmetric and positive semi-definite, this Loewner ordering is equivalent to bounding its maximum eigenvalue by $L$
$$
\nabla^2 f_\mu(\mathbf{X}) \preceq L I \iff \lambda_{\max}(\nabla^2 f_\mu(\mathbf{X})) \leq L
$$
From the bound obtained in **(j-2)**, we know that
$$
\lambda_{\max}(\nabla^2 f_\mu(\mathbf{X})) \leq \sum_{i=1}^n \|\mathbf{a}_i\|_2^2 + \mu
$$
Recalling the definition of the Frobenius norm of a matrix $\mathbf{A} \in \mathbb{R}^{n \times p}$ whose $i$-th row is $\mathbf{a}_i^\top$, the sum of the squared $\ell_2$-norms of its rows is precisely the squared Frobenius norm
$$
\sum_{i=1}^n \|\mathbf{a}_i\|_2^2 = \sum_{i=1}^n \sum_{j=1}^p a_{ij}^2 = \|\mathbf{A}\|_F^2
$$
Substituting this identity into the upper bound gives
$$
\lambda_{\max}(\nabla^2 f_\mu(\mathbf{X})) \leq \|\mathbf{A}\|_F^2 + \mu = L
$$
which concludes the proof.

__(l)__ (3 points) KL divergence and NLL. Let $q_i=(q(c\mid\mathbf{a}_i))_{c=1}^C$ be the true conditional label distribution and $\mathbf{p}_i$ the model softmax distribution. Write $\mathrm{KL}(q_i\,\|\,\mathbf{p}_i)$ and show that minimizing its average over the samples is equivalent to minimizing the expected average negative log-likelihood, where the expectation is over labels drawn from $q$ with the features held fixed. Explain how $f(\mathbf{X})/n$ from (a) estimates this expected loss.

Let $q_i = (q(c \mid \mathbf{a}_i))_{c=1}^C$ be the true conditional distribution of the label given the features $\mathbf{a}_i$, and $\mathbf{p}_i = (p(c \mid \mathbf{a}_i))_{c=1}^C$ the predicted softmax distribution, which depends on the model parameters $\mathbf{X}$.
The KL divergence is defined as:
$$
\begin{align*}
\mathrm{KL}(q_i \parallel \mathbf{p}_i) 
&= \sum_{c=1}^C q(c \mid \mathbf{a}_i) \log \left( \frac{q(c \mid \mathbf{a}_i)}{p(c \mid \mathbf{a}_i)} \right) \\
&= \sum_{c=1}^C q(c \mid \mathbf{a}_i) \log q(c \mid \mathbf{a}_i) - \sum_{c=1}^C q(c \mid \mathbf{a}_i) \log p(c \mid \mathbf{a}_i) \\
&= -H(q_i) + H(q_i, \mathbf{p}_i)
\end{align*}
$$
where $H(q_i)$ is the entropy of the true distribution and $H(q_i, \mathbf{p}_i)$ is the cross-entropy.

The entropy $H(q_i)$ depends only on the true distribution, so it is constant with respect to $\mathbf{X}$ and we can drop it from the argmin:
$$
\arg\min_{\mathbf{X}} \frac{1}{n} \sum_{i=1}^n \mathrm{KL}(q_i \parallel \mathbf{p}_i) = \arg\min_{\mathbf{X}} \frac{1}{n} \sum_{i=1}^n \left[ - \sum_{c=1}^C q(c \mid \mathbf{a}_i) \log p(c \mid \mathbf{a}_i) \right]
$$

In general we do not know $q_i$: what we observe is only the label $b_i$, which is one sample drawn from $q_i$. However, the cross-entropy can be read as an expectation. It is exactly the expected negative log-likelihood of a label drawn from $q_i$:
$$
- \sum_{c=1}^C q(c \mid \mathbf{a}_i) \log p(c \mid \mathbf{a}_i) = \mathbb{E}_{b_i \sim q_i}\left[ - \log p(b_i \mid \mathbf{a}_i) \right]
$$

Keeping the features fixed and assuming the labels $b_1, \dots, b_n$ are drawn independently, by linearity of expectation the average over all $n$ samples gives:
$$
\frac{1}{n} \sum_{i=1}^n H(q_i, \mathbf{p}_i) = \mathbb{E}_{b_1, \dots, b_n}\left[ \frac{1}{n} \sum_{i=1}^n - \log p(b_i \mid \mathbf{a}_i) \right] = \mathbb{E}_{b_1, \dots, b_n}\left[ \frac{1}{n} f(\mathbf{X}) \right]
$$
where $f(\mathbf{X})$ is the function derived in part (a). So minimizing the average KL divergence is equivalent to minimizing the expected average NLL, since the two objectives only differ by the constant $\frac{1}{n} \sum_{i=1}^n H(q_i)$.

Of course we cannot compute this expectation, because we only have the labels we actually observed. What we can compute is $\frac{1}{n} f(\mathbf{X})$ itself, which is an unbiased estimate of the expected loss, since $\mathbb{E}\left[ \frac{1}{n} f(\mathbf{X}) \right] = \frac{1}{n} \sum_{i=1}^n H(q_i, \mathbf{p}_i)$ for every fixed $\mathbf{X}$. Moreover, since it is an average of $n$ independent terms, its variance is
$$
\mathrm{Var}\left( \frac{1}{n} f(\mathbf{X}) \right) = \frac{1}{n^2} \sum_{i=1}^n \mathrm{Var}_{q_i}\left( \log p(b_i \mid \mathbf{a}_i) \right) = \mathcal{O}(1/n),
$$
assuming the log-probabilities are bounded. So by the law of large numbers, $\frac{1}{n} f(\mathbf{X})$ gets closer and closer to the expected loss as $n$ grows.

This shows that minimizing the empirical average NLL $\frac{1}{n} f(\mathbf{X})$ approximately minimizes the average KL divergence between the true conditional distribution and the model's predictions.

The unregularized maximum likelihood problem may have no finite minimizer. Adding a quadratic regularizer with $\mu>0$ gives a unique regularized estimate, defined by the smooth, strongly convex problem:
<a id="eq-regularized-estimator"></a>

$$
 \mathbf{X}^\star = \arg\min_{\mathbf{X} \in \mathbb{R}^{p\times C}} f(\mathbf{X}) + \frac{\mu}{2}\|\mathbf{X}\|_F^2.\tag{2}
$$


## Binary logistic regression (specialization for Part 2)

While this part analyzed the multiclass (softmax) setting, in the next exercise we will continue under the simplified two-class case.

Let labels be $b_i \in \{-1, +1\}$, features $\mathbf{a}_i \in \mathbb{R}^p$, and weight vector $\mathbf{x} \in \mathbb{R}^p$. Define the sigmoid
$$
\sigma(t) = \frac{1}{1+e^{-t}}.
$$
Model the conditional distribution as
$$
\mathbb{P}(b_i = j \mid \mathbf{a}_i) = \sigma\big(j\, \mathbf{a}_i^\top \mathbf{x}\big), \quad j \in \{-1,+1\}.
$$
The likelihood over i.i.d. samples $\{(\mathbf{a}_i, b_i)\}_{i=1}^n$ is
$$
\mathcal{L}(\mathbf{x}) = \prod_{i=1}^n \sigma\big(b_i\, \mathbf{a}_i^\top \mathbf{x}\big),
$$
so the negative log-likelihood is
$$
 f(\mathbf{x}) = -\log \mathcal{L}(\mathbf{x}) = \sum_{i=1}^n \log\big(1 + e^{-b_i\, \mathbf{a}_i^\top \mathbf{x}}\big).
$$

__(m)__ (2 points) Show that the gradient of the negative log-likelihood is the standard binary logistic regression gradient:
$$
\nabla f(\mathbf{x}) = \sum_{i=1}^n \big(-b_i\, \sigma(-b_i\, \mathbf{a}_i^\top \mathbf{x})\big)\, \mathbf{a}_i.
$$
(Hint: use the chain rule and $\sigma'(t) = \sigma(t)\big(1-\sigma(t)\big)$.)

We will use this binary formulation in Part 2 - First order methods.


1Let $z_i = b_i \mathbf{a}_i^\top \mathbf{x}$. Using the definition of the sigmoid function $\sigma(t) = \frac{1}{1 + e^{-t}}$, we can rewrite $f(\mathbf{x})$ using $\sigma(z_i)$ as follows
$$
f(\mathbf{x}) = \sum_{i=1}^n \log\left(1 + e^{-b_i \mathbf{a}_i^\top \mathbf{x}}\right) = \sum_{i=1}^n \log\left(\frac{1}{\sigma(z_i)}\right) = \sum_{i=1}^n -\log \sigma(z_i)
$$

By applying the chain rule and the given identity $\sigma'(t) = \sigma(t)(1 - \sigma(t))$, we have
$$
\begin{align*}
\nabla (-\log \sigma(z_i)) &= -\frac{\sigma'(z_i)}{\sigma(z_i)} \nabla_{\mathbf{x}} z_i \\
&= -\frac{\sigma(z_i)(1 - \sigma(z_i))}{\sigma(z_i)} (b_i \mathbf{a}_i) \\
&= -b_i (1 - \sigma(z_i)) \mathbf{a}_i
\end{align*}
$$

Now notice that
$$
\begin{align*}
1 - \sigma(z_i) 
&= 1 - \frac{1}{1 + e^{-z_i}} \\
&= \frac{e^{-z_i}}{1 + e^{-z_i}} \\ 
&= \frac{1}{1 + e^{z_i}} = \sigma(-z_i) \\
&= \sigma(-b_i \mathbf{a}_i^\top \mathbf{x}) \\
\implies \nabla f(\mathbf{x}) &= \sum_{i=1}^n \left( -b_i \sigma(-b_i \mathbf{a}_i^\top \mathbf{x}) \right) \mathbf{a}_i
\end{align*}
$$
and we have concluded.